# **Phase 1: Data Foundation**

This notebook loads the raw Seoul bike file, cleans it with the functions in `src/clean.py`, validates it, splits open and closed hours and saves two clean files.

Outputs: `data/clean/clean.csv` (open hours) and `data/clean/closed_hours.csv` (closed hours).

## **Step 0: Import all Necessary Libraries**

In [1]:
%load_ext autoreload
%autoreload 2

import re
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")

from src.clean import (
    CLEAN_FILE,
    CLOSED_FILE,
    CLOSED_LABEL,
    HOURS_PER_DAY,
    MAX_HUMIDITY,
    add_calendar_features,
    load_clean,
    load_raw,
    parse_date_column,
    save_clean,
    split_closed,
    standardise_columns,
    validate,
)

RAW_FILE = "../data/raw/SeoulBikeData.csv"
CLEAN_DIR = Path("../data/clean")

## **Step 1. Load the raw file and standardise column names**

In [2]:
df = load_raw(RAW_FILE)
display(df.head())
print(df.shape)
print(df.columns.tolist())

st = standardise_columns(df)
print(st.columns.tolist())

EXPECTED_COLUMNS = [
    "date", "rented_bike_count", "hour", "temperature", "humidity",
    "wind_speed", "visibility", "dew_point_temperature", "solar_radiation",
    "rainfall", "snowfall", "seasons", "holiday", "functioning_day",
]

found_columns = st.columns.tolist()
assert found_columns == EXPECTED_COLUMNS, (
    f"Column names do not match the expected list. Got {found_columns}"
)

invalid_columns = [col for col in found_columns if not re.fullmatch(r"[a-z_]+", col)]
assert not invalid_columns, (
    f"Columns hold characters other than lowercase letters and underscores: {invalid_columns}"
)

,Date,Rented Bike Count,Hour,Temperature(?),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(?),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


(8760, 14)
['Date', 'Rented Bike Count', 'Hour', 'Temperature(?)', 'Humidity(%)', 'Wind speed (m/s)', 'Visibility (10m)', 'Dew point temperature(?)', 'Solar Radiation (MJ/m2)', 'Rainfall(mm)', 'Snowfall (cm)', 'Seasons', 'Holiday', 'Functioning Day']
['date', 'rented_bike_count', 'hour', 'temperature', 'humidity', 'wind_speed', 'visibility', 'dew_point_temperature', 'solar_radiation', 'rainfall', 'snowfall', 'seasons', 'holiday', 'functioning_day']


## **Step 2. Parse the date column**

In [3]:
st = parse_date_column(st)

min_date = st["date"].min()
max_date = st["date"].max()
unique_dates = st["date"].nunique()
print(f"Minimum date: {min_date}")
print(f"Maximum date: {max_date}")
print(f"Number of unique dates: {unique_dates}")

EXPECTED_MIN_DATE = pd.Timestamp("2017-12-01")
EXPECTED_MAX_DATE = pd.Timestamp("2018-11-30")
EXPECTED_UNIQUE_DATES = 365

assert pd.api.types.is_datetime64_any_dtype(st["date"]), (
    f"Date column has dtype {st['date'].dtype}, expected datetime"
)
assert min_date == EXPECTED_MIN_DATE, (
    f"Minimum date is {min_date}, expected {EXPECTED_MIN_DATE}"
)
assert max_date == EXPECTED_MAX_DATE, (
    f"Maximum date is {max_date}, expected {EXPECTED_MAX_DATE}"
)
assert unique_dates == EXPECTED_UNIQUE_DATES, (
    f"Found {unique_dates} unique dates, expected {EXPECTED_UNIQUE_DATES}"
)

Minimum date: 2017-12-01 00:00:00
Maximum date: 2018-11-30 00:00:00
Number of unique dates: 365


## **Step 3. Add calendar features**

In [25]:
st = add_calendar_features(st)

display(
    st[["date", "year", "month", "weekday", "is_weekend", "week_position", "week_number"]]
    .drop_duplicates()
    .sort_values(by="date")
    .head(7)
)
print(st.shape)

EXPECTED_YEARS = {2017, 2018}
EXPECTED_MONTHS = 12

years = set(st["year"].unique())
assert years == EXPECTED_YEARS, f"Found years {years}, expected {EXPECTED_YEARS}"

months = st["month"].nunique()
assert months == EXPECTED_MONTHS, f"Found {months} months, expected {EXPECTED_MONTHS}"

# 365 days = 52 full weeks (364 days) + 1 extra day
# 52 weeks * 2 weekend days = 104
# The extra day is the last date, 2018-11-30, a Friday, so it adds 0 weekend dates
EXPECTED_WEEKEND_DATES = 104
weekend_dates = st.loc[st["is_weekend"], "date"].dt.date.nunique()
assert weekend_dates == EXPECTED_WEEKEND_DATES, (
    f"Found {weekend_dates} weekend dates, expected {EXPECTED_WEEKEND_DATES}"
)

# 2017-12-01 is a Friday and sits at week position 0
# 2017-12-07 is a Thursday and sits at week position 6
EXPECTED_POSITION_FRIDAY = 0
EXPECTED_POSITION_THURSDAY = 6
position_friday = st.loc[st["date"] == pd.Timestamp("2017-12-01"), "week_position"].iloc[0]
position_thursday = st.loc[st["date"] == pd.Timestamp("2017-12-07"), "week_position"].iloc[0]
assert position_friday == EXPECTED_POSITION_FRIDAY, (
    f"2017-12-01 has week_position {position_friday}, expected {EXPECTED_POSITION_FRIDAY}"
)
assert position_thursday == EXPECTED_POSITION_THURSDAY, (
    f"2017-12-07 has week_position {position_thursday}, expected {EXPECTED_POSITION_THURSDAY}"
)

# 1. 2017 12 01 and 2017 12 07
# (0 // 7) + 1 = 1
# (6 // 7) + 1 = 1

EXPECTED = [1, 1]
DATES = pd.to_datetime(["2017-12-01", "2017-12-07"])

GOT = (
    st.loc[st["date"].isin(DATES), ["date", "week_number"]]
    .drop_duplicates(subset="date")["week_number"]
    .tolist()
)

assert GOT == EXPECTED, f"expected {EXPECTED}, got {GOT}"


# 2. 2017 12 08
# (7 // 7) + 1 = 2

EXPECTED = 2
DATE = pd.Timestamp("2017-12-08")

GOT = st.loc[st["date"] == DATE, "week_number"].iloc[0]

assert GOT == EXPECTED, f"expected {EXPECTED}, got {GOT}"


# 3. 2017 12 28
# (27 // 7) + 1 = 4

EXPECTED = 4
DATE = pd.Timestamp("2017-12-28")

GOT = st.loc[st["date"] == DATE, "week_number"].iloc[0]

assert GOT == EXPECTED, f"expected {EXPECTED}, got {GOT}"


# 4. 2017 12 29 and 2017 12 31
# (28 // 7) + 1 = 5
# (30 // 7) + 1 = 5

EXPECTED = [5, 5]
DATES = pd.to_datetime(["2017-12-29", "2017-12-31"])

GOT = (
    st.loc[st["date"].isin(DATES), ["date", "week_number"]]
    .drop_duplicates(subset="date")["week_number"]
    .tolist()
)

assert GOT == EXPECTED, f"expected {EXPECTED}, got {GOT}"


# 5. 2018 11 30
# (364 // 7) + 1 = 53

EXPECTED = 53
DATE = pd.Timestamp("2018-11-30")

GOT = st.loc[st["date"] == DATE, "week_number"].iloc[0]

assert GOT == EXPECTED, f"expected {EXPECTED}, got {GOT}"

,date,year,month,weekday,is_weekend,week_position,week_number
0,2017-12-01,2017,12,4,False,0,1
24,2017-12-02,2017,12,5,True,1,1
48,2017-12-03,2017,12,6,True,2,1
72,2017-12-04,2017,12,0,False,3,1
96,2017-12-05,2017,12,1,False,4,1
120,2017-12-06,2017,12,2,False,5,1
144,2017-12-07,2017,12,3,False,6,1


(8760, 21)


## **Step 4. Validate the full file**

Validation runs before the split because the rules (8,760 rows, 24 rows per date, unique date and hour pairs) hold on the full file.

In [5]:
validate(st)
print("validate passed")

validate passed


## **Step 5. Split open and closed hours**

In [6]:
open_df, closed_df = split_closed(st)

EXPECTED_OPEN_ROWS = 8465           # functioning_day == "Yes" value counts
EXPECTED_CLOSED_ROWS = 295          # functioning_day == "No" value counts
EXPECTED_CLOSED_RENTALS = 0
EXPECTED_SHARED_LABELS = 0
EXPECTED_OPEN_HAS_CLOSED_HOURS = False

# 2017 holds 31 dates (December). 2018 January to November holds 365 - 31 = 334 dates.
# 12 whole closed days leave open_df. The partial day 2018-10-06 keeps 17 open hours and stays.
# 365 - 12 = 353
EXPECTED_OPEN_DATES = 353

open_rows = len(open_df)
closed_rows = len(closed_df)

# Assert 1: row counts
assert open_rows == EXPECTED_OPEN_ROWS, (
    f"Open rows: expected {EXPECTED_OPEN_ROWS}, got {open_rows}"
)
assert closed_rows == EXPECTED_CLOSED_ROWS, (
    f"Closed rows: expected {EXPECTED_CLOSED_ROWS}, got {closed_rows}"
)

# Assert 2: the two parts rebuild the full file
total_rows = open_rows + closed_rows
assert total_rows == len(st), (
    f"Open plus closed rows: expected {len(st)}, got {total_rows}"
)

# Assert 3: closed hours hold no rentals
closed_rental_sum = closed_df["rented_bike_count"].sum()
assert closed_rental_sum == EXPECTED_CLOSED_RENTALS, (
    f"Closed rentals: expected {EXPECTED_CLOSED_RENTALS}, got {closed_rental_sum}"
)

# Assert 4a: no closed hour sits in open_df
open_has_closed_hours = (open_df["functioning_day"] == CLOSED_LABEL).any()
assert open_has_closed_hours == EXPECTED_OPEN_HAS_CLOSED_HOURS, (
    f"Closed hours in open_df: expected {EXPECTED_OPEN_HAS_CLOSED_HOURS}, got {open_has_closed_hours}"
)

# Assert 4b: the two parts share no row labels
shared_labels = len(open_df.index.intersection(closed_df.index))
assert shared_labels == EXPECTED_SHARED_LABELS, (
    f"Shared row labels: expected {EXPECTED_SHARED_LABELS}, got {shared_labels}"
)

# Assert 5: unique dates in open_df
open_dates = open_df["date"].nunique()
assert open_dates == EXPECTED_OPEN_DATES, (
    f"Open dates: expected {EXPECTED_OPEN_DATES}, got {open_dates}"
)

### **Closed hours per date**

In [7]:
closed_hours_per_date = closed_df.groupby("date").size()
display(closed_hours_per_date)

EXPECTED_CLOSED_DATES = 13
EXPECTED_PARTIAL_HOURS = [7]
EXPECTED_CLOSED_YEARS = {2018}

closed_dates = len(closed_hours_per_date)
assert closed_dates == EXPECTED_CLOSED_DATES, (
    f"Closed dates: expected {EXPECTED_CLOSED_DATES}, got {closed_dates}"
)

partial_hours = closed_hours_per_date[closed_hours_per_date != HOURS_PER_DAY].tolist()
assert partial_hours == EXPECTED_PARTIAL_HOURS, (
    f"Partial closed days: expected {EXPECTED_PARTIAL_HOURS}, got {partial_hours}"
)

closed_years = set(closed_df["year"].unique())
assert closed_years == EXPECTED_CLOSED_YEARS, (
    f"Closed years: expected {EXPECTED_CLOSED_YEARS}, got {closed_years}"
)

date
2018-04-11    24
2018-05-10    24
2018-09-18    24
2018-09-19    24
2018-09-28    24
2018-09-30    24
2018-10-02    24
2018-10-04    24
2018-10-06     7
2018-10-09    24
2018-11-03    24
2018-11-06    24
2018-11-09    24
dtype: int64

The table lists 13 closed dates. 12 of them hold 24 hours each and one holds 7 hours, which is a partial closure on 2018-10-06. All closed dates fall in 2018 and none fall in December 2017, so my 2017 report needs no adjustment. Daily averages keep 2018-10-06 (17 open hours) with a footnote.

## **Step 6. Save the clean files**

In [8]:
save_clean(open_df, closed_df, CLEAN_DIR)

## **Step 7. Reload and check**

In [9]:
# A plain read keeps the date as text, because a CSV stores text and holds no dtype information
clean_plain = pd.read_csv(CLEAN_DIR / CLEAN_FILE)
print(clean_plain["date"].dtype)
plain_is_datetime = pd.api.types.is_datetime64_any_dtype(clean_plain["date"])
assert not plain_is_datetime, "Plain read_csv parsed the date column, expected text"

# load_clean restores the date column as datetime
clean_reload = load_clean(CLEAN_DIR / CLEAN_FILE)
closed_reload = load_clean(CLEAN_DIR / CLOSED_FILE)
print(clean_reload["date"].dtype)
print(closed_reload["date"].dtype)

# Row counts
reload_clean_rows = len(clean_reload)
assert reload_clean_rows == EXPECTED_OPEN_ROWS, (
    f"Clean reload rows: expected {EXPECTED_OPEN_ROWS}, got {reload_clean_rows}"
)
reload_closed_rows = len(closed_reload)
assert reload_closed_rows == EXPECTED_CLOSED_ROWS, (
    f"Closed reload rows: expected {EXPECTED_CLOSED_ROWS}, got {reload_closed_rows}"
)

# Date dtypes
reload_clean_dtype = clean_reload["date"].dtype
assert pd.api.types.is_datetime64_any_dtype(clean_reload["date"]), (
    f"Clean reload date dtype: expected datetime, got {reload_clean_dtype}"
)
reload_closed_dtype = closed_reload["date"].dtype
assert pd.api.types.is_datetime64_any_dtype(closed_reload["date"]), (
    f"Closed reload date dtype: expected datetime, got {reload_closed_dtype}"
)

# Columns: index=False means no extra index column appears after the reload
reload_clean_columns = clean_reload.columns.tolist()
assert reload_clean_columns == open_df.columns.tolist(), (
    f"Clean reload columns {reload_clean_columns} differ from open_df columns"
)
reload_closed_columns = closed_reload.columns.tolist()
assert reload_closed_columns == closed_df.columns.tolist(), (
    f"Closed reload columns {reload_closed_columns} differ from closed_df columns"
)

object
datetime64[ns]
datetime64[ns]


## **Step 8. Break tests**

Each test feeds `validate` a damaged copy of the data and proves the right check fails.

In [10]:
# Break test 1: drop one row, so check 1 (row count) must fail
broken_rows = st.iloc[1:]
try:
    validate(broken_rows)
except AssertionError as error:
    print(f"Check 1 failed as expected: {error}")
else:
    raise RuntimeError("validate did not fail on a missing row")

Check 1 failed as expected: Row count is 8759, expected 8760


In [11]:
# Break test 2: humidity one above the maximum, so check 6 must fail
broken_humidity = st.assign(humidity=MAX_HUMIDITY + 1)
try:
    validate(broken_humidity)
except AssertionError as error:
    print(f"Check 6 failed as expected: {error}")
else:
    raise RuntimeError("validate did not fail on humidity above the maximum")

Check 6 failed as expected: Largest humidity is 101, expected at most 100


## **Step 9. Folder check**

In [12]:
PROJECT_ROOT = Path("..")
EXPECTED_FOLDERS = ["data/raw", "data/clean", "notebooks", "src"]
IGNORED_PARTS = {".ipynb_checkpoints", ".git", "__pycache__", ".venv", ".vscode"}

for path in sorted(PROJECT_ROOT.rglob("*")):
    if IGNORED_PARTS.isdisjoint(path.parts):
        print(path.relative_to(PROJECT_ROOT).as_posix())

missing_folders = [folder for folder in EXPECTED_FOLDERS if not (PROJECT_ROOT / folder).is_dir()]
assert not missing_folders, f"Missing folders: {missing_folders}"

data
data/clean
data/clean/clean.csv
data/clean/closed_hours.csv
data/raw
data/raw/SeoulBikeData.csv
notebooks
notebooks/01_data_foundation.ipynb
notebooks/02_analysis_2017.ipynb
src
src/analysis.py
src/clean.py


## **Phase 1 notes**

**Why the date column is text after a plain read?**

A CSV stores text and holds no information about data types. Hence a plain read_csv leaves the date column in clean_plain as text.

**Index choice. I set index to False because**

The reload then holds no extra unnamed column. The cost is the raw row labels. After a reload the labels restart at 0, so a row cannot be traced back to the raw file by position. Date and hour still identify each row.

**Why load_clean exists?**

Without load_clean, a notebook reading the file with a plain read_csv gets the date column as text. Then the .dt accessor raises an AttributeError, and sorting and comparing treat dates as text. The function keeps the rule, parse_dates=["date"], in one place.

**Why closed_df is kept?**

I keep closed_df because the report must name the excluded dates and because open and closed rows together rebuild the full file.